In [ ]:
# datasets name : Flowers Recognition 

import pandas as pd
import numpy as np
import os
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from albumentations import ToTensorV2
import albumentations as A
import random

import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import Adam
from torchmetrics.classification import MulticlassF1Score, MulticlassAccuracy,MulticlassRecall


def seed_everything(seed: int = 42):
    random.seed(seed)          # python random
    np.random.seed(seed)       # numpy
    torch.manual_seed(seed)    # torch CPU
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

base_path=Path(r"/kaggle/input/datasets/alxmamaev/flowers-recognition/flowers")
path_list=[]

for img_path in base_path.rglob("*.jpg"):
    path_list.append({"label":img_path.parent.name,"path":img_path})

df=pd.DataFrame(path_list)
df['targets']=pd.factorize(df['label'])[0]
df=df.sample(frac=1,random_state=42).reset_index(drop=True)

train_df,tmp_df=train_test_split(df,test_size=0.3,stratify=df['label'],random_state=42)
val_df,test_df=train_test_split(tmp_df,test_size=0.4,stratify=tmp_df['label'],random_state=42)

img_aug=A.Compose([
    A.RandomResizedCrop(size=(224,224),scale=(0.8,1.0),ratio=(0.9,1.1),p=1),
    A.HorizontalFlip(p=0.3),
    A.Affine(scale=(0.9,1.1),rotate=(-15,15),border_mode=cv2.BORDER_REFLECT_101,p=0.3),
    A.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2,hue=0.03,p=0.6),
    A.CoarseDropout(num_holes_range=(1, 1), hole_height_range=(48, 48),
                    hole_width_range=(48, 48),fill=0,p=0.25)
])

tr_resnet34=A.Compose([
    A.RandomResizedCrop(size=(224,224),scale=(0.8,1.0),ratio=(0.9,1.1),p=1),
    A.HorizontalFlip(p=0.3),
    A.Affine(scale=(0.9,1.1),rotate=(-15,15),border_mode=cv2.BORDER_REFLECT_101,p=0.3),
    A.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2,hue=0.03,p=0.6),
    A.CoarseDropout(num_holes_range=(1, 1), hole_height_range=(48, 48),
                    hole_width_range=(48, 48),fill=0,p=0.25),
    A.Normalize(mean=(0.485, 0.456, 0.406),std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_resnet34=A.Compose([
    A.Resize(224,224,p=1),
    A.Normalize(mean=(0.485, 0.456, 0.406),std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class FlowerCustom(Dataset):
    def __init__(self,path,targets,augment=None):
        self.path=path
        self.targets=targets
        self.augment=augment
    def __len__(self):
        return len(self.path)
    def __getitem__(self,idx):
        img=cv2.imread(self.path[idx])
        img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
        if self.augment is None:
            raise ValueError("IMG Augment must be need")
        img=self.augment(image=img)['image']
        targets=torch.tensor(self.targets[idx],dtype=torch.long)
        return img,targets


train_custom=FlowerCustom(train_df['path'].to_list(),train_df['targets'].to_list(),
                          augment=tr_resnet34)
val_custom=FlowerCustom(val_df['path'].to_list(),val_df['targets'].to_list(),
                        augment=val_resnet34)
test_custom=FlowerCustom(test_df['path'].to_list(),test_df['targets'].to_list(),
                         augment=val_resnet34)

train_loader=DataLoader(train_custom,batch_size=32,shuffle=True,num_workers=4,pin_memory=True)
val_loader=DataLoader(val_custom,batch_size=32,shuffle=False,num_workers=4,pin_memory=True)
test_loader=DataLoader(test_custom,batch_size=32,shuffle=False,num_workers=4,pin_memory=True)

print("done")

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layer=nn.Sequential(
            nn.Conv2d(3,64,kernel_size=3,stride=1,padding=1),
            nn.LeakyReLU(0.1),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,kernel_size=3,stride=1,padding=1),
            nn.LeakyReLU(0.1),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2),

            nn.Conv2d(128,256,kernel_size=3,stride=1,padding=1),
            nn.LeakyReLU(0.1),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1,1)) 
        )

        self.fc_layer=nn.Sequential(
            nn.Linear(256,128),
            nn.LeakyReLU(0.1),
            nn.BatchNorm1d(128),
            nn.Linear(128,5),
            nn.LogSoftmax(dim=-1)
        )

    def forward(self,x):
        x=self.conv_layer(x)
        x=torch.flatten(x,1)
        x=self.fc_layer(x)
        return x

    def forward_logit(self,model,x): # logit값만 구하는 로직
        feats=model.conv_layer(x)
        feats=torch.flatten(feats,1) # 0번 차원(B)은 그대로 두고, 1번 차원부터 끝까지 flatten하라는것. (B,1152)
        logits=model.fc_layer[:-1](feats) # 시퀀셜 인덱싱. LogSoftmax전까지만 슬라이싱
        return logits

from typing import List
from dataclasses import dataclass,field
from tqdm import tqdm

@dataclass
class History:
    training_accuracy:List[float]=field(default_factory=list)
    training_recall:List[float]=field(default_factory=list)
    training_loss:List[float]=field(default_factory=list)
    val_accuracy:List[float]=field(default_factory=list)
    val_recall:List[float]=field(default_factory=list)
    val_loss:List[float]=field(default_factory=list)
history=History()


class Trainer:
    def __init__(self,train_loader,val_loader,model,optimizer,loss_func,
                 scheduler,metric_acc,metric_rec,device,history,mode="min"):
        self.model=model
        self.train_loader=train_loader
        self.val_loader=val_loader
        self.optimizer=optimizer
        self.loss_func=loss_func
        self.scheduler=scheduler
        self.metric_acc=metric_acc
        self.metric_rec=metric_rec
        self.device=device
        self.history=history
        if mode=="max":
            self.best_value=float('-inf')
        else:
            self.best_value=float('inf')

    def training_epoch(self,epoch):
        self.metric_acc.reset()
        self.metric_rec.reset()
        self.model.train()
        loss_sum=0.0
        avg_loss=0.0
        with tqdm(total=len(self.train_loader),desc=f"training {epoch}",leave=True) as bar:
            for batch_idx,(x_train,y_train) in enumerate(self.train_loader):
                x_train=x_train.to(self.device)
                y_train=y_train.to(self.device)
                logits=self.model(x_train)
                loss=self.loss_func(logits,y_train)
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                loss_sum+=loss.item()
                avg_loss=loss_sum/(batch_idx+1)
                preds=logits.argmax(dim=1)   # dim=-1과 같다. (B,12)  1번 dim 즉 행에 대해서
                self.metric_acc.update(preds, y_train)
                self.metric_rec.update(preds, y_train)
                bar.update(1)

                if batch_idx%10==0:
                    acc=self.metric_acc.compute().item()
                    recall=self.metric_rec.compute().item()
                    bar.set_postfix({"acc": acc, "recall":recall, "loss":avg_loss,"epoch":epoch})
            return self.metric_acc.compute().item(), self.metric_rec.compute().item(),avg_loss  

    def validating_epoch(self,epoch):
        self.metric_acc.reset()
        self.metric_rec.reset()
        self.model.eval()
        loss_sum=0
        avg_loss=0.0
        with tqdm(total=len(self.val_loader),desc=f"validating {epoch}", leave=True) as bar:
            with torch.no_grad():
                for batch_idx,(x_val,y_val) in enumerate(self.val_loader):
                    x_val=x_val.to(self.device)
                    y_val=y_val.to(self.device)
                    logits=self.model(x_val)
                    loss=self.loss_func(logits,y_val)

                    preds=logits.argmax(dim=-1)
                    self.metric_acc.update(preds,y_val)
                    self.metric_rec.update(preds,y_val)
                    loss_sum+=loss.item()
                    avg_loss=loss_sum/(batch_idx+1)
                    bar.update(1)
                    if batch_idx%10==0:
                        acc=self.metric_acc.compute().item()
                        recall=self.metric_rec.compute().item()
                        bar.set_postfix({"acc": acc, "recall":recall, "loss":avg_loss,"epoch":epoch})
                return self.metric_acc.compute().item(), self.metric_rec.compute().item(),avg_loss

    
    def fit(self,epochs,early_stop,path):
        stop_count=0   
        for epoch in range(epochs):
            training_accuracy,training_recall,training_loss=self.training_epoch(epoch)
            self.history.training_accuracy.append(training_accuracy)
            self.history.training_recall.append(training_recall)
            self.history.training_loss.append(training_loss)
            val_accuracy,val_recall,val_loss=self.validating_epoch(epoch)
            self.history.val_accuracy.append(val_accuracy)
            self.history.val_recall.append(val_recall)
            self.history.val_loss.append(val_loss)
            
            if self.best_value>val_loss:
                self.best_value=val_loss
                stop_count=0
                torch.save(self.model.state_dict(),os.path.join(path,f"{epoch}_{val_loss}.pt"))
            else:
                stop_count+=1
                if stop_count>=early_stop:
                    print(f"early_stopped. current epoch : {epoch}")
                    return self.history
                    
        return self.history

from torch.optim import Adam
from torchmetrics import Accuracy,Recall
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os

device="cuda" if torch.cuda.is_available() else "cpu"
model=SimpleCNN()
model=model.to(device)
optimizer=Adam(model.parameters(),lr=1e-3)
loss_func = nn.NLLLoss()
acc_metric=Accuracy(task="multiclass",num_classes=5)
recall_metric=Recall(task="multiclass",num_classes=5,average="macro")
scheduler=ReduceLROnPlateau(optimizer,factor=0.1,patience=3)
model=model.to(device)
metric_rec=recall_metric.to(device)
metric_acc=acc_metric.to(device)

output_path=r"/kaggle/working/"
t=Trainer(train_loader,val_loader,model,optimizer,loss_func,scheduler,acc_metric,recall_metric,device,history,mode="min")
history=t.fit(30,5,output_path)

In [ ]:
from sklearn.metrics import confusion_matrix

best_param=torch.load(r"/kaggle/working/28_0.5584180355072021.pt")
model.load_state_dict(best_param)

class Predict:
    def __init__(self,model,test_loader,device):
        self.model=model.to(device)
        self.test_loader=test_loader
        self.device=device
        self.actual_list=[]
        self.pred_list=[]
    def predict(self):
        with tqdm(total=len(self.test_loader),desc=f"predicting",leave=True) as bar:
            self.model.eval()
            for x,y in self.test_loader:
                x=x.to(self.device)
                y=y.to(self.device)
                logits=self.model(x)
                pred=torch.argmax(logits,dim=-1)
                self.pred_list.extend(pred.detach().cpu().numpy().tolist())
                self.actual_list.extend(y.detach().cpu().numpy().tolist())
                bar.update(1)
            return self.actual_list,self.pred_list

p=Predict(model,test_loader,device)
answer,preds=p.predict()
cm=confusion_matrix(answer,preds)

### Feature Map
>[32, 256, 56, 56]을 conv연산에서 얻었다면 256개 각각의 채널은 56x56 feature_map을 가지고 있다
>
>56,56에서 0,0 좌상단을 위치이다. 이것은 모든 256개의 채널의 H,W에서 동일
>
>채널에는 좌표가 없다(단지 특징종류만 있을뿐)

In [ ]:
# 주석제거
import torch.nn.functional as F

class GradCAM5:
    def __init__(self,model,idx):
        self.model=model
        self.target_layer=self.model.conv_layer[idx]  # idx 10으로 가정
        self.feature_map=None
        self.gradient=None
        self.fwd_handle=self.target_layer.register_forward_hook(self._forward)
        self.bwd_handle=self.target_layer.register_full_backward_hook(self._backward)
    def remove(self):
        self.fwd_handle.remove()
        self.bwd_handle.remove()
    def _forward(self,module,inputs,outputs):
        self.feature_map=outputs
    def _backward(self,module,grad_inputs,grad_outputs):
        self.gradient=grad_outputs[0]
    @torch.no_grad()
    def _normalize(self,cam):                                    # 입력된 cam shape (16,1,224,224)
        batch_size=cam.shape[0]                                     
        cam_flatten=cam.view(batch_size,-1)                      # (16,1*224*224) => (16, 50176) 
        cam_min=cam_flatten.min(dim=1)[0].view(batch_size,1,1,1) # (16,).view(batch_size,1,1,1) => (16,1,1,1)
        cam_max=cam_flatten.max(dim=1)[0].view(batch_size,1,1,1) # (16,).view(batch_size,1,1,1) => (16,1,1,1)
        cam=(cam-cam_min)/(cam_max-cam_min+1e-8)                 # broadcasting => (16, 1, 224, 224)
        return cam                                               # batch 16장의 Grad-CAM
    def generate(self,x,class_idx=None):    # x가 (16,3,224,224).  num_classes는 5로 가정. 
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        logits=self.model(x)                      # (16, 5)
        preds=logits.argmax(dim=-1)               # (16,)
        batch_size=logits.shape[0]
        if class_idx is None:
            target_idx=preds                       # (16,)
        elif isinstance(class_idx,int):
            target_idx=torch.full((batch_size,),class_idx,device=logits.device,dtype=torch.long)
        else:
            target_idx=class_idx.to(logits.device)
        score_each=logits.gather(1,target_idx.unsqueeze(1)).squeeze(1)  # (16,)
        # target_idx.unsqueeze(1) : (16,1)
        # logits.gather(dim, index) : logits <= (16,5)  index(target_idx.unsqueeze(1)) <= (16,1)
        # score_each <= (16,1) 이것을 squeeze(1)하니깐 최종적으로 (16,)

        score=score_each.sum()
        score.backward()
        grad=self.gradient                         # 후킹한곳의 shape는 (16,256,56,56)
        act=self.feature_map                       # (16,256,56,56)   
        weights=grad.mean(dim=(2,3),keepdim=True)  # (16,256,1,1)
        cam=(act*weights).sum(dim=1,keepdim=True)  # (16,1,56,56) <= 브로드캐스팅 되고 dim 1을 sum후 keep
        cam=F.relu(cam)
        cam=F.interpolate(cam,size=(x.shape[2],x.shape[3]),mode='bilinear',align_corners=False) # (16,1,224,224)
        cam=cam.detach()
        cam=self._normalize(cam) #_normalize 호출. 입력 shape는 (16,1,224,224) 반환값도 => (16, 1, 224, 224) 
        return cam.cpu(), class_idx, float(score.cpu().detach())

In [ ]:
images, labels = next(iter(test_loader))
images.shape

x=images[0:4].to(device)
y=labels[0:4]

x.shape

In [ ]:
import matplotlib.pyplot as plt

def visualizer(images,nrows=4,ncols=3):
    if images.shape[0]!= nrows:
        print("이미지가 부족합니다")
        return False
    gradcam=GradCAM5(model,idx=10)
    cam,pred_idx,score=gradcam.generate(images)
    gradcam.remove()
    images=images.detach().cpu()
    cam=cam.detach().cpu()
    fig,axes=plt.subplots(nrows=nrows,ncols=ncols,figsize=(nrows*3,ncols*3))
    
    for i in range(4):
        img=images[i].permute(1,2,0)
        cmp=cam[i].permute(1,2,0)
        # 원본
        axes[i,0].imshow(img)
        axes[i,0].set_title("Original")
        axes[i,0].axis("off")
    
        # CAM
        axes[i,1].imshow(cmp, cmap="jet")
        axes[i,1].set_title("Grad-CAM")
        axes[i,1].axis("off")
    
        # Overlay
        axes[i,2].imshow(img)
        axes[i,2].imshow(cmp, cmap="jet", alpha=0.45)
        axes[i,2].set_title("Overlay")
        axes[i,2].axis("off")
    
    plt.tight_layout()
    plt.show()


visualizer(x)
    
    
    

In [ ]:
from torchinfo import summary

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layer=nn.Sequential(
            nn.Conv2d(3,64,kernel_size=3,stride=1,padding=1),
            nn.LeakyReLU(0.1),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,kernel_size=3,stride=1,padding=1),
            nn.LeakyReLU(0.1),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2),

            nn.Conv2d(128,256,kernel_size=3,stride=1,padding=1),
            nn.LeakyReLU(0.1),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1,1)) 
        )

        self.fc_layer=nn.Sequential(
            nn.Linear(256,128),
            nn.LeakyReLU(0.1),
            nn.BatchNorm1d(128),
            nn.Linear(128,5),
            nn.LogSoftmax(dim=-1)
        )
    def forward(self,x):
        x=self.conv_layer(x)
        x=torch.flatten(x,1)
        x=self.fc_layer(x)
        return x        



test_class=SimpleCNN()

input_data=torch.randn(32,3,224,224)
summary(test_class,input_size=(32, 3, 224, 224),col_names=["input_size","output_size","num_params"],
        row_settings=['var_names'])